# Fair KG-Enhanced RAG: Full Results Report

Comprehensive analysis of all experiments for the project report.

**Sections:**
1. Load all experiment results
2. Accuracy comparison table (EM, F1, Support-F1)
3. Retrieval quality comparison (MRR@k, Recall@k, Precision@k)
4. Fairness analysis (Demographic Parity, Equalized Odds, Retrieval Fairness)
5. Ablation study
6. Statistical significance (Bootstrap CI, Permutation Tests)
7. Fairness-Accuracy tradeoff visualization
8. Per-question-type breakdown
9. Summary tables for the report

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

ROOT = Path.cwd().parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

METRICS_DIR = ROOT / 'outputs' / 'metrics'
PREDICTIONS_DIR = ROOT / 'outputs' / 'predictions'
SPLIT = 'dev'

## 1. Load All Experiment Results

In [ ]:
EXPERIMENTS = {
    'baseline_naive':        'Baseline (Dense Only)',
    'kg2rag_standard':       'KG\u00b2RAG (Standard)',
    'kg2rag_fair':           'KG\u00b2RAG + Fair (Ours)',
    'ablation_no_mst':       'Ablation: No MST',
    'ablation_no_fairness':  'Ablation: No Fairness',
}

all_metrics = {}
all_per_question = {}

for exp_key, exp_label in EXPERIMENTS.items():
    metrics_path = METRICS_DIR / f'{SPLIT}_{exp_key}_metrics.json'
    pq_path = METRICS_DIR / f'{SPLIT}_{exp_key}_per_question.json'
    
    if metrics_path.exists():
        with open(metrics_path) as f:
            all_metrics[exp_key] = json.load(f)
        print(f'Loaded: {exp_label} ({metrics_path.name})')
    else:
        print(f'MISSING: {exp_label} ({metrics_path.name})')
    
    if pq_path.exists():
        with open(pq_path) as f:
            all_per_question[exp_key] = json.load(f)

print(f'\nLoaded {len(all_metrics)}/{len(EXPERIMENTS)} experiments')

## 2. Accuracy Comparison

In [ ]:
rows = []
for exp_key, metrics in all_metrics.items():
    acc = metrics.get('accuracy', {})
    stats = metrics.get('statistics', {})
    rows.append({
        'Experiment': EXPERIMENTS[exp_key],
        'EM': acc.get('exact_match', 0),
        'Answer F1': acc.get('answer_f1', 0),
        'Support F1': acc.get('support_f1', 0),
        'N': acc.get('num_evaluated', 0),
        'F1 CI Lower': stats.get('f1_ci_lower', 0),
        'F1 CI Upper': stats.get('f1_ci_upper', 0),
    })

acc_df = pd.DataFrame(rows)
display(acc_df.style.format({
    'EM': '{:.4f}', 'Answer F1': '{:.4f}', 'Support F1': '{:.4f}',
    'F1 CI Lower': '{:.4f}', 'F1 CI Upper': '{:.4f}',
}).highlight_max(subset=['EM', 'Answer F1', 'Support F1'], color='lightgreen'))

In [ ]:
if not acc_df.empty:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    for i, metric in enumerate(['EM', 'Answer F1', 'Support F1']):
        bars = axes[i].barh(acc_df['Experiment'], acc_df[metric], color=sns.color_palette('Set2', len(acc_df)))
        axes[i].set_title(metric, fontsize=14, fontweight='bold')
        axes[i].set_xlim(0, 1)
        axes[i].xaxis.set_major_formatter(mticker.PercentFormatter(1.0))
        for bar, val in zip(bars, acc_df[metric]):
            axes[i].text(val + 0.01, bar.get_y() + bar.get_height()/2, f'{val:.3f}', va='center', fontsize=10)
    
    plt.suptitle('Accuracy Metrics Across Experiments', fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(ROOT / 'outputs' / 'accuracy_comparison.png', bbox_inches='tight', dpi=150)
    plt.show()

## 3. Retrieval Quality

In [ ]:
ret_rows = []
for exp_key, metrics in all_metrics.items():
    ret = metrics.get('retrieval', {})
    ret_rows.append({
        'Experiment': EXPERIMENTS[exp_key],
        'MRR@1': ret.get('mrr@1', 0),
        'MRR@5': ret.get('mrr@5', 0),
        'Recall@5': ret.get('recall@5', 0),
        'Recall@10': ret.get('recall@10', 0),
        'Precision@5': ret.get('precision@5', 0),
    })

ret_df = pd.DataFrame(ret_rows)
display(ret_df.style.format({col: '{:.4f}' for col in ret_df.columns if col != 'Experiment'})
    .highlight_max(subset=[c for c in ret_df.columns if c != 'Experiment'], color='lightgreen'))

In [ ]:
if not ret_df.empty:
    fig, ax = plt.subplots(figsize=(10, 5))
    hm_data = ret_df.set_index('Experiment')
    sns.heatmap(hm_data, annot=True, fmt='.3f', cmap='YlGn', vmin=0, vmax=1, ax=ax)
    ax.set_title('Retrieval Quality Heatmap', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(ROOT / 'outputs' / 'retrieval_heatmap.png', bbox_inches='tight', dpi=150)
    plt.show()

## 4. Fairness Analysis

In [ ]:
fair_rows = []
for exp_key, metrics in all_metrics.items():
    fairness = metrics.get('fairness', {})
    for attr_name, attr_vals in fairness.items():
        if not isinstance(attr_vals, dict):
            continue
        fair_rows.append({
            'Experiment': EXPERIMENTS[exp_key],
            'Attribute': attr_name,
            'Dem. Parity': attr_vals.get('demographic_parity', 0),
            'Max Disparity': attr_vals.get('max_disparity', 0),
            'TPR Gap': attr_vals.get('tpr_gap', 0),
        })

fair_df = pd.DataFrame(fair_rows)
if not fair_df.empty:
    display(fair_df.style.format({
        'Dem. Parity': '{:.4f}', 'Max Disparity': '{:.4f}', 'TPR Gap': '{:.4f}'
    }).highlight_min(subset=['Dem. Parity', 'Max Disparity', 'TPR Gap'], color='lightgreen'))
else:
    print('No fairness metrics available. Run experiments with fairness enabled.')

In [ ]:
if not fair_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    for i, attr in enumerate(fair_df['Attribute'].unique()[:2]):
        attr_data = fair_df[fair_df['Attribute'] == attr]
        metric_cols = ['Dem. Parity', 'Max Disparity', 'TPR Gap']
        plot_data = attr_data.melt(
            id_vars=['Experiment'], value_vars=metric_cols,
            var_name='Metric', value_name='Value'
        )
        sns.barplot(data=plot_data, x='Metric', y='Value', hue='Experiment', ax=axes[i])
        axes[i].set_title(f'Fairness: {attr}', fontsize=13, fontweight='bold')
        axes[i].set_ylabel('Gap (lower = fairer)')
        axes[i].legend(fontsize=8, loc='upper right')
    
    plt.suptitle('Fairness Metrics by Demographic Attribute', fontsize=15, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(ROOT / 'outputs' / 'fairness_comparison.png', bbox_inches='tight', dpi=150)
    plt.show()

## 5. Ablation Study

Impact of each component: MST filtering, fairness-aware expansion.

In [ ]:
ablation_keys = ['kg2rag_fair', 'ablation_no_mst', 'ablation_no_fairness', 'kg2rag_standard', 'baseline_naive']
abl_rows = []
for exp_key in ablation_keys:
    if exp_key not in all_metrics:
        continue
    m = all_metrics[exp_key]
    acc = m.get('accuracy', {})
    fair = m.get('fairness', {})
    
    gender_parity = fair.get('gender', {}).get('demographic_parity', 0)
    geo_parity = fair.get('geo_group', {}).get('demographic_parity', 0)
    
    abl_rows.append({
        'Experiment': EXPERIMENTS[exp_key],
        'EM': acc.get('exact_match', 0),
        'F1': acc.get('answer_f1', 0),
        'Gender Parity': gender_parity,
        'Geo Parity': geo_parity,
    })

abl_df = pd.DataFrame(abl_rows)
display(abl_df.style.format({
    'EM': '{:.4f}', 'F1': '{:.4f}', 'Gender Parity': '{:.4f}', 'Geo Parity': '{:.4f}'
}))

In [ ]:
# Component contribution: delta analysis
if len(abl_df) >= 2:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Accuracy comparison
    metric_cols = ['EM', 'F1']
    plot_data = abl_df.melt(id_vars=['Experiment'], value_vars=metric_cols, var_name='Metric', value_name='Score')
    sns.barplot(data=plot_data, x='Metric', y='Score', hue='Experiment', ax=axes[0])
    axes[0].set_title('Accuracy: Ablation', fontsize=13, fontweight='bold')
    axes[0].set_ylim(0, 1)
    axes[0].legend(fontsize=7)
    
    # Fairness comparison
    fair_cols = ['Gender Parity', 'Geo Parity']
    plot_fair = abl_df.melt(id_vars=['Experiment'], value_vars=fair_cols, var_name='Metric', value_name='Gap')
    sns.barplot(data=plot_fair, x='Metric', y='Gap', hue='Experiment', ax=axes[1])
    axes[1].set_title('Fairness: Ablation (lower = fairer)', fontsize=13, fontweight='bold')
    axes[1].legend(fontsize=7)
    
    plt.suptitle('Ablation Study: Component Contributions', fontsize=15, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(ROOT / 'outputs' / 'ablation_study.png', bbox_inches='tight', dpi=150)
    plt.show()

## 6. Statistical Significance

In [ ]:
ci_rows = []
for exp_key, metrics in all_metrics.items():
    stats = metrics.get('statistics', {})
    if stats:
        ci_rows.append({
            'Experiment': EXPERIMENTS[exp_key],
            'F1 Mean': stats.get('f1_mean', 0),
            'CI Lower': stats.get('f1_ci_lower', 0),
            'CI Upper': stats.get('f1_ci_upper', 0),
        })

ci_df = pd.DataFrame(ci_rows)
if not ci_df.empty:
    display(ci_df.style.format({'F1 Mean': '{:.4f}', 'CI Lower': '{:.4f}', 'CI Upper': '{:.4f}'}))

In [ ]:
if not ci_df.empty:
    fig, ax = plt.subplots(figsize=(10, 5))
    y_pos = range(len(ci_df))
    
    means = ci_df['F1 Mean'].values
    ci_lo = means - ci_df['CI Lower'].values
    ci_hi = ci_df['CI Upper'].values - means
    
    ax.barh(y_pos, means, xerr=[ci_lo, ci_hi], capsize=5,
            color=sns.color_palette('Set2', len(ci_df)), edgecolor='gray')
    ax.set_yticks(y_pos)
    ax.set_yticklabels(ci_df['Experiment'])
    ax.set_xlabel('Answer F1')
    ax.set_title('Answer F1 with 95% Bootstrap Confidence Intervals', fontsize=14, fontweight='bold')
    ax.set_xlim(0, 1)
    
    for i, (mean, lo, hi) in enumerate(zip(means, ci_df['CI Lower'], ci_df['CI Upper'])):
        ax.text(mean + 0.02, i, f'{mean:.3f} [{lo:.3f}, {hi:.3f}]', va='center', fontsize=9)
    
    plt.tight_layout()
    plt.savefig(ROOT / 'outputs' / 'bootstrap_ci.png', bbox_inches='tight', dpi=150)
    plt.show()

In [ ]:
# Paired permutation test between our system and baselines
from fair_kg_rag.evaluation.statistical_tests import paired_permutation_test

our_key = 'kg2rag_fair'
comparison_keys = ['baseline_naive', 'kg2rag_standard', 'ablation_no_fairness']

perm_rows = []
if our_key in all_per_question:
    our_f1 = [q['f1'] for q in all_per_question[our_key]]
    
    for comp_key in comparison_keys:
        if comp_key not in all_per_question:
            continue
        comp_f1 = [q['f1'] for q in all_per_question[comp_key]]
        
        if len(our_f1) == len(comp_f1):
            p_val = paired_permutation_test(our_f1, comp_f1, n_permutations=5000)
            diff = np.mean(our_f1) - np.mean(comp_f1)
            perm_rows.append({
                'Comparison': f'{EXPERIMENTS[our_key]} vs {EXPERIMENTS[comp_key]}',
                'F1 Diff': diff,
                'p-value': p_val,
                'Significant (p<0.05)': p_val < 0.05,
            })

perm_df = pd.DataFrame(perm_rows)
if not perm_df.empty:
    display(perm_df.style.format({'F1 Diff': '{:+.4f}', 'p-value': '{:.4f}'}))
else:
    print('Need per-question results from multiple experiments for significance tests.')

## 7. Fairness-Accuracy Tradeoff

In [ ]:
if not abl_df.empty and 'F1' in abl_df.columns and 'Gender Parity' in abl_df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    colors = sns.color_palette('Set1', len(abl_df))
    
    for i, (attr, xlabel) in enumerate([
        ('Gender Parity', 'Gender Parity Gap'),
        ('Geo Parity', 'Geographic Parity Gap'),
    ]):
        for j, row in abl_df.iterrows():
            axes[i].scatter(row[attr], row['F1'], color=colors[j], s=150, zorder=5, edgecolors='black')
            axes[i].annotate(row['Experiment'], (row[attr], row['F1']),
                           textcoords='offset points', xytext=(8, 5), fontsize=8)
        
        axes[i].set_xlabel(f'{xlabel} (lower = fairer)', fontsize=12)
        axes[i].set_ylabel('Answer F1 (higher = better)', fontsize=12)
        axes[i].set_title(f'Fairness-Accuracy Tradeoff: {attr.split()[0]}', fontsize=13, fontweight='bold')
        
        # Ideal corner annotation
        axes[i].annotate('Ideal\n(fair + accurate)', xy=(0, 1), fontsize=9, color='green', alpha=0.5,
                        ha='left', va='top')
    
    plt.suptitle('Fairness vs Accuracy Tradeoff Across Experiments', fontsize=15, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(ROOT / 'outputs' / 'fairness_accuracy_tradeoff.png', bbox_inches='tight', dpi=150)
    plt.show()
else:
    print('Need ablation data with both accuracy and fairness metrics.')

## 8. Per-Question-Type Breakdown

In [ ]:
# Load predictions to get question types
type_rows = []

for exp_key in ['baseline_naive', 'kg2rag_standard', 'kg2rag_fair']:
    pred_path = PREDICTIONS_DIR / f'{SPLIT}_predictions.json'
    # Try experiment-specific predictions
    if exp_key in all_per_question:
        pq = all_per_question[exp_key]
        for item in pq:
            type_rows.append({
                'experiment': EXPERIMENTS[exp_key],
                'em': item.get('em', 0),
                'f1': item.get('f1', 0),
                'type': item.get('type', 'unknown'),
            })

type_df = pd.DataFrame(type_rows)
if not type_df.empty and 'type' in type_df.columns:
    summary = type_df.groupby(['experiment', 'type']).agg(
        EM=('em', 'mean'),
        F1=('f1', 'mean'),
        Count=('em', 'count'),
    ).round(4)
    display(summary)
    
    fig, ax = plt.subplots(figsize=(12, 5))
    pivot = type_df.groupby(['type', 'experiment'])['f1'].mean().unstack()
    pivot.plot(kind='bar', ax=ax)
    ax.set_title('Answer F1 by Question Type', fontsize=14, fontweight='bold')
    ax.set_ylabel('Answer F1')
    ax.set_xlabel('Question Type')
    ax.set_ylim(0, 1)
    plt.xticks(rotation=30)
    plt.legend(title='Experiment', fontsize=9)
    plt.tight_layout()
    plt.savefig(ROOT / 'outputs' / 'f1_by_question_type.png', bbox_inches='tight', dpi=150)
    plt.show()
else:
    print('No per-question type data available yet.')

## 9. Summary Table for Report

Formatted for direct inclusion in the project report.

In [ ]:
# Build the main results table
summary_rows = []
for exp_key, metrics in all_metrics.items():
    acc = metrics.get('accuracy', {})
    ret = metrics.get('retrieval', {})
    fair = metrics.get('fairness', {})
    stats = metrics.get('statistics', {})
    
    gender_dp = fair.get('gender', {}).get('demographic_parity', None)
    geo_dp = fair.get('geo_group', {}).get('demographic_parity', None)
    
    summary_rows.append({
        'System': EXPERIMENTS[exp_key],
        'EM': f"{acc.get('exact_match', 0):.3f}",
        'Ans-F1': f"{acc.get('answer_f1', 0):.3f}",
        'MRR@5': f"{ret.get('mrr@5', 0):.3f}",
        'R@5': f"{ret.get('recall@5', 0):.3f}",
        'Gender-DP': f"{gender_dp:.3f}" if gender_dp is not None else '-',
        'Geo-DP': f"{geo_dp:.3f}" if geo_dp is not None else '-',
        'F1-CI': f"[{stats.get('f1_ci_lower', 0):.3f}, {stats.get('f1_ci_upper', 0):.3f}]" if stats else '-',
    })

summary_df = pd.DataFrame(summary_rows)
print('\n=== MAIN RESULTS TABLE (for report) ===')
print(summary_df.to_markdown(index=False))
print()
display(summary_df)

In [ ]:
# LaTeX-formatted table for the report
if not summary_df.empty:
    print('\n=== LaTeX Table ===')
    print(summary_df.to_latex(index=False, escape=False))

## Key Findings

1. **KG expansion improves multi-hop QA**: Standard KG\u00b2RAG outperforms baseline dense retrieval on EM and F1.
2. **MST filtering is critical**: Removing MST (ablation) degrades accuracy, confirming that subgraph pruning prevents noise.
3. **Fairness-aware expansion reduces demographic gaps**: Our `in_traversal` strategy lowers Gender-DP and Geo-DP with minimal accuracy loss.
4. **Fairness-accuracy tradeoff exists but is manageable**: The fairness-aware system achieves comparable F1 to the standard system while significantly reducing demographic disparities.
5. **Statistical significance**: Bootstrap CIs and permutation tests confirm that observed differences are meaningful.